In [1]:
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

from embedder import Embedder

# Notebook lives in rag/build-samples/, corpus in <root>/corpus/
ROOT = Path.cwd().resolve()
while not (ROOT / "corpus" / "chunks").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
CHUNKS_PATH = ROOT / "corpus" / "chunks" / "time_chunks.json"
VECTORS_PATH = CHUNKS_PATH.with_suffix(".npy")            # time_chunks.npy
META_PATH = CHUNKS_PATH.with_suffix(".meta.json")         # time_chunks.meta.json

chunks = json.loads(CHUNKS_PATH.read_text(encoding="utf-8"))
print(f"{len(chunks)} chunks from {CHUNKS_PATH}")

embedder = Embedder(batch_size=8)     # BAAI/bge-m3 from Hugging Face; GPU + fp16 if CUDA is available
print("model:", embedder.model_name, "| device:", embedder.device, "| fp16:", embedder.fp16, "| dim:", embedder.dim)

917 chunks from S:\ai_projects\demo_26_RAG-vs-agent\corpus\chunks\time_chunks.json


S:\ai_projects\demo_26_RAG-vs-agent\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 24423.98it/s]


model: BAAI/bge-m3 | device: cpu | fp16: False | dim: 1024


S:\ai_projects\demo_26_RAG-vs-agent\rag\build-samples\embedder.py:69: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return int(self.model.get_sentence_embedding_dimension())


In [2]:
# Row i of the matrix = embedding of chunks[i] (same order as the JSON, no ids stored inside the .npy).
# Plain chunk text, no title/date header: the vectors will be clustered, so they must reflect content only.
vectors = embedder.embed([c["text"] for c in chunks])

np.save(VECTORS_PATH, vectors)        # float32, shape (n_chunks, dim)
META_PATH.write_text(json.dumps({
    "model": embedder.model_name,
    "backend": "sentence-transformers",
    "device": embedder.device,
    "fp16": embedder.fp16,
    "dim": int(vectors.shape[1]),
    "count": int(vectors.shape[0]),
    "normalized": embedder.normalize,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "ids": [c["id"] for c in chunks],  # guard against the JSON being re-chunked after embedding
}, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"{vectors.shape} {vectors.dtype} -> {VECTORS_PATH}  ({VECTORS_PATH.stat().st_size / 1e6:.1f} MB)")

Batches: 100%|██████████| 115/115 [30:15<00:00, 15.78s/it]

(917, 1024) float32 -> S:\ai_projects\demo_26_RAG-vs-agent\corpus\chunks\time_chunks.npy  (3.8 MB)


In [4]:
# ---- Sanity checks: reload from disk and verify alignment with the JSON ----
V = np.load(VECTORS_PATH)
meta = json.loads(META_PATH.read_text(encoding="utf-8"))
chunks = json.loads(CHUNKS_PATH.read_text(encoding="utf-8"))

assert V.shape == (len(chunks), meta["dim"]), f"shape {V.shape} != ({len(chunks)}, {meta['dim']})"
assert V.dtype == np.float32
assert np.isfinite(V).all(), "NaN/inf in vectors"
assert meta["ids"] == [c["id"] for c in chunks], "chunks.json no longer matches the vectors - re-embed"

norms = np.linalg.norm(V, axis=1)
print(f"shape {V.shape}, norms min/max: {norms.min():.4f} / {norms.max():.4f}")

# nearest neighbour from ANOTHER stream: a quick look at whether vectors carry topical meaning
sims = V @ V.T
stream = np.array([c["youtubeId"] for c in chunks])
for i in (0, len(chunks) // 3, 2 * len(chunks) // 3):
    s = np.where(stream == stream[i], -1.0, sims[i])
    j = int(s.argmax())
    print(f"\n[{chunks[i]['id']}] {chunks[i]['text'][:160]}...")
    print(f"  ~ {s[j]:.3f} [{chunks[j]['id']}] {chunks[j]['text'][:160]}...")

shape (917, 1024), norms min/max: 1.0000 / 1.0000

[wht8lf-h9N8#00:00:00] І так, з чого б хотілось би почати? Почнемо ми з того, що привітаюся, да. Мене звати Олег, якщо хто не знає, якщо там хтось перший раз дивиться або перший раз н...
  ~ 0.723 [R5IXxa1e8K4#00:00:00] Сьогодні у нас п'ятниця. Зараз у нас уже 8:00 за Лондоном і 10:00 за Києвом. Сьогодні у нас друга зустріч, перша була така більш демонстраційна. А, я себе не по...

[YDs0AGzYjoI#00:04:07] нічого не придумав, тому можна просто поздоровити мене з проше з минулим днем народженням, тому да, отак. Поздравляем. Да, отак. Да. Отак, будем творчих успіхів...
  ~ 0.731 [R5IXxa1e8K4#00:00:00] Сьогодні у нас п'ятниця. Зараз у нас уже 8:00 за Лондоном і 10:00 за Києвом. Сьогодні у нас друга зустріч, перша була така більш демонстраційна. А, я себе не по...

[v9dLbeQk3YY#00:21:43] ніякого немає. Банер. На банері можна додати додаткову продаючу інформацію про себе, да, яку ось сюди не влізло. Як працюють ці всі штучки? Наша увага почин